In [1]:
### Set parameters

# model_type_str = "1_multil"
# model_type_str = "2_sonoisa"
# model_type_str = "3_tohoku_bbwwm"
# model_type_str = "3_tohoku_bb2"
model_type_str = "4_izumil"

model_name_str = ""
if model_type_str == "1_multil":
    model_name_str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
elif model_type_str == "2_sonoisa":
    model_name_str = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"    
elif model_type_str == "3_tohoku_bbwwm":
    model_name_str = "cl-tohoku/bert-base-japanese-whole-word-masking"        
elif model_type_str == "3_tohoku_bb2":
    model_name_str = "cl-tohoku/bert-base-japanese-v2"        
elif model_type_str == "4_izumil":
    model_name_str = "izumi-lab/bert-small-japanese"        


print(f"{model_type_str = }")
print(f"{model_name_str = }")

model_type_str = '4_izumil'
model_name_str = 'izumi-lab/bert-small-japanese'


In [2]:
import glob

filepaths_input_str_list = glob.glob("../../datasets/jppost/local_govt_lists/0*")
filepaths_input_str_list.sort()


filepaths_input_str_dict = dict()

for filepath_str in filepaths_input_str_list:
    type_str = filepath_str.replace("../../datasets/jppost/local_govt_lists/", "").replace("_list.txt", "")
    filepaths_input_str_dict[type_str] = filepath_str

print(filepaths_input_str_dict)

{'00_prefectures': '../../datasets/jppost/local_govt_lists/00_prefectures_list.txt', '01_shi': '../../datasets/jppost/local_govt_lists/01_shi_list.txt', '02_ku': '../../datasets/jppost/local_govt_lists/02_ku_list.txt', '03_gun': '../../datasets/jppost/local_govt_lists/03_gun_list.txt', '04_cho': '../../datasets/jppost/local_govt_lists/04_cho_list.txt', '05_son': '../../datasets/jppost/local_govt_lists/05_son_list.txt'}


In [3]:
local_govt_lists_dict = dict()

for type_str, filepath_str in filepaths_input_str_dict.items():
    local_govt_list = list()
    
    with open(filepath_str) as fp:
        for line in fp:
            local_govt_list.append(line.rstrip())
    
    local_govt_lists_dict[type_str] = local_govt_list
    

print(local_govt_lists_dict)

{'00_prefectures': ['北海道', '青森県', '岩手県', '宮城県', '秋田県', '山形県', '福島県', '茨城県', '栃木県', '群馬県', '埼玉県', '千葉県', '東京都', '神奈川県', '新潟県', '富山県', '石川県', '福井県', '山梨県', '長野県', '岐阜県', '静岡県', '愛知県', '三重県', '滋賀県', '京都府', '大阪府', '兵庫県', '奈良県', '和歌山県', '鳥取県', '島根県', '岡山県', '広島県', '山口県', '徳島県', '香川県', '愛媛県', '高知県', '福岡県', '佐賀県', '長崎県', '熊本県', '大分県', '宮崎県', '鹿児島県', '沖縄県'], '01_shi': ['韮崎市', '小野市', '京都市', '野田市', '弘前市', '東御市', '日高市', '山形市', '多賀城市', '市川市', '防府市', '宝塚市', '高知市', 'つくば市', '太宰府市', '八街市', '銚子市', '仙北市', '狛江市', '三郷市', '飯田市', '小城市', '安中市', '入間市', '明石市', '会津若松市', '柏崎市', '阿久根市', '男鹿市', '長野市', '中野市', '佐野市', '南相馬市', 'あきる野市', '小松市', '旭市', '伊那市', '室蘭市', '三島市', '小千谷市', '小樽市', '姫路市', '尼崎市', '小牧市', '羽村市', '笠岡市', '高梁市', '鴨川市', '岸和田市', '羽生市', '大津市', '倉敷市', '摂津市', '横浜市', '砺波市', '近江八幡市', '大村市', '印西市', '南房総市', '小諸市', '三笠市', '阿蘇市', '旭川市', '清瀬市', '都留市', '飯能市', '安芸市', '鶴岡市', '四條畷市', '夕張市', '杵築市', '泉南市', '大和高田市', '塩竈市', '沼津市', '南島原市', '豊見城市', '八幡平市', '水俣市', '赤平市', 'むつ市', '松原市', '一宮市', '曽於市', '高槻市', '三浦市', '亀岡市', '富岡市', '

In [ ]:
from sentence_transformers import SentenceTransformer, models


transformer = models.Transformer(model_name_str)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling])


In [5]:
import csv


def tokenize_and_to_csv(model, sentences_list, filepath_output_str):
    st_tokenizer = model._first_module().tokenizer
    input_ids = model._first_module().tokenize(sentences_list)['input_ids']
    
    with open(filepath_output_str, mode='w', encoding='utf-8') as fp:
        for input_id in input_ids:
            write = csv.writer(fp)
            write.writerow(st_tokenizer.convert_ids_to_tokens(input_id, skip_special_tokens = True))


for type_str, local_govt_list in local_govt_lists_dict.items():
    filepath_output_str = "tokenized_local_govt_names_without_special_tokens/tokens_" + type_str + "_" + model_type_str + ".csv"
    tokenize_and_to_csv(model, local_govt_list, filepath_output_str)


In [6]:
if model_type_str == "1_multil":
    unk_token_str = "<unk>"
else:
    unk_token_str = "[UNK]"


def find_unk_tokens(model, sentences_list, filepath_output_str, unk_token_str):
    st_tokenizer = model._first_module().tokenizer
    input_ids = model._first_module().tokenize(sentences_list)['input_ids']

    tokenized_sentences_list = list()
    max_colnum = 0
    unk_sentence_count = 0

    # Create a tokenized sentences list and find the maximum number of tokens in a sentence
    for n, input_id in enumerate(input_ids):
        tokens_list = st_tokenizer.convert_ids_to_tokens(input_id)
        if unk_token_str in tokens_list:
            tokenized_sentences_list.append([sentences_list[n] + ': '] + tokens_list)
            max_colnum = max(max_colnum, len(tokens_list))
            unk_sentence_count += 1
    
    print(unk_sentence_count)

    # Write the tokenized sentences list on a file as csv
    max_cols_list = list(range(1, max_colnum + 1))
    with open(filepath_output_str, mode='w', encoding='utf-8') as fp:
        write = csv.writer(fp)
        
        if tokenized_sentences_list:
            write.writerow(max_cols_list)
            
            for tokenized_sentence in tokenized_sentences_list:
                write.writerow(tokenized_sentence)
        else:
            write.writerow(['no unks'])


for type_str, local_govt_list in local_govt_lists_dict.items():
    filepath_output_str = "tokenized_local_govt_names_with_unknown_tokens/tokens_" + type_str + "_" + model_type_str + ".csv"
    find_unk_tokens(model, local_govt_list, filepath_output_str, unk_token_str)



0
0
0
0
0
0
